<a href="https://colab.research.google.com/github/asiahcrutchfield/ground-level-languages-ai/blob/ai-audio/gll_intro_to_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# Ground Level Languages — Accurate Acoustic Phoneme Pipeline
# Models: Torchaudio MMS_FA (Forced Alignment) + TorchCREPE Pitch Tracking
# ==============================================================================

!pip install -q torchaudio torchcrepe librosa soundfile scipy numpy

import os
import json
import zipfile
import numpy as np
import torch
import torchaudio
import torchcrepe
import librosa
import soundfile as sf
from scipy.signal import find_peaks

# 1. Setup directories & device
os.makedirs("audio/slices", exist_ok=True)
os.makedirs("audio/phonemes", exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1b. Download authentic Oxford IPA reference phonemes (true uncoarticulated oral posture)
OXFORD_PHONEMES = {
    "k.wav": "k_isolation.mp3",
    "ae.wav": "%C3%A6_isolation.mp3",
    "t.wav": "t_isolation.mp3",
    "d.wav": "d_isolation.mp3",
    "ao.wav": "%C9%92_isolation.mp3",
    "g.wav": "g_isolation.mp3",
    "w.wav": "w_isolation.mp3",
    "dz.wav": "d%CA%92_isolation.mp3",
    "ah.wav": "%CA%8C_isolation.mp3",
    "m.wav": "m_isolation.mp3",
    "p.wav": "p_isolation.mp3",
    "r.wav": "r_isolation.mp3",
    "iy.wav": "i%CB%90_isolation.mp3",
    "l.wav": "l_isolation.mp3",
    "aa.wav": "%C9%91%CB%90_isolation.mp3"
}

import urllib.request, ssl, subprocess
_ctx = ssl._create_unverified_context()
_base_url = "https://raw.githubusercontent.com/xiaozhah/phoneme_audio/main/audio/"

print("Checking authentic Oxford phoneme database...")
for wav_name, remote_file in OXFORD_PHONEMES.items():
    out_path = f"audio/phonemes/{wav_name}"
    if not os.path.exists(out_path):
        tmp_mp3 = f"/content/tmp_{wav_name}.mp3"
        url = _base_url + remote_file
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, context=_ctx, timeout=8) as res:
                with open(tmp_mp3, "wb") as f:
                    f.write(res.read())
            subprocess.run(["ffmpeg", "-y", "-i", tmp_mp3, "-ar", "16000", "-ac", "1", out_path], capture_output=True)
            print(f"  -> Installed {wav_name}")
        except Exception as e:
            print(f"  -> Note: could not fetch {wav_name}: {e}")
        finally:
            if os.path.exists(tmp_mp3): os.remove(tmp_mp3)

# Auto-sync audio files if running in fresh Colab session
REPO_URL = "https://github.com/asiahcrutchfield/ground-level-languages-ai.git"
if not os.path.exists("audio/cat.mp3") and not os.path.exists("/content/audio/cat.mp3") and not os.path.exists("/content/cat.mp3"):
    print("Fetching audio assets from repository...")
    !git clone --depth 1 $REPO_URL repo_temp
    if os.path.exists("repo_temp/audio"):
        !cp -r repo_temp/audio .
        !cp -r repo_temp/images . 2>/dev/null || true
        !cp repo_temp/assessment_data.json . 2>/dev/null || true
    !rm -rf repo_temp

# 2. Canonical Phonetic Lexicon for Level 0 Words
# Guarantees all phonemes (like /æ/ in 'cat') are present in exact chronological order
WORD_PHONEME_LEXICON = {
    "cat": [
        {"ipa": "k",  "kind": "consonant", "char": "c", "ref_audio": "audio/phonemes/k.wav"},
        {"ipa": "æ",  "kind": "vowel",     "char": "a", "ref_audio": "audio/phonemes/ae.wav"},
        {"ipa": "t",  "kind": "consonant", "char": "t", "ref_audio": "audio/phonemes/t.wav"}
    ],
    "dog": [
        {"ipa": "d",  "kind": "consonant", "char": "d", "ref_audio": "audio/phonemes/d.wav"},
        {"ipa": "ɔ",  "kind": "vowel",     "char": "o", "ref_audio": "audio/phonemes/ao.wav"},
        {"ipa": "ɡ",  "kind": "consonant", "char": "g", "ref_audio": "audio/phonemes/g.wav"}
    ],
    "walk": [
        {"ipa": "w",  "kind": "consonant", "char": "w", "ref_audio": "audio/phonemes/w.wav"},
        {"ipa": "ɔ",  "kind": "vowel",     "char": "a"},
        {"ipa": "k",  "kind": "consonant", "char": "k"}
    ],
    "jump": [
        {"ipa": "dʒ", "kind": "consonant", "char": "j", "ref_audio": "audio/phonemes/dz.wav"},
        {"ipa": "ʌ",  "kind": "vowel",     "char": "u", "ref_audio": "audio/phonemes/ah.wav"},
        {"ipa": "m",  "kind": "consonant", "char": "m", "ref_audio": "audio/phonemes/m.wav"},
        {"ipa": "p",  "kind": "consonant", "char": "p", "ref_audio": "audio/phonemes/p.wav"}
    ],
    "tree": [
        {"ipa": "t",  "kind": "consonant", "char": "t", "ref_audio": "audio/phonemes/t.wav"},
        {"ipa": "ɹ",  "kind": "consonant", "char": "r", "ref_audio": "audio/phonemes/r.wav"},
        {"ipa": "iː", "kind": "vowel",     "char": "e", "ref_audio": "audio/phonemes/iy.wav"}
    ]
}

# 3. Load Torchaudio MMS_FA Forced Aligner (aligns speech audio to phoneme tokens)
bundle = torchaudio.pipelines.MMS_FA
fa_model = bundle.get_model().to(device)
fa_tokenizer = bundle.get_tokenizer()
fa_aligner = bundle.get_aligner()

def apply_fade(clip, sr=16000, fade_ms=10):
    fade_len = min(int(sr * (fade_ms / 1000.0)), len(clip) // 2)
    if fade_len <= 1:
        return clip
    win = np.ones(len(clip), dtype=np.float32)
    win[:fade_len] = np.linspace(0.0, 1.0, fade_len)
    win[-fade_len:] = np.linspace(1.0, 0.0, fade_len)
    return clip * win

def synthesize_humming_track(y, f0, periodicity, sr=16000, hop_length=160):
    total_samples = len(y)
    freq_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(f0)), f0)
    conf_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(periodicity)), periodicity)
    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=hop_length)[0]
    rms_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(rms)), rms)
    rms_norm = rms_interp / (np.max(rms_interp) + 1e-6)
    voiced_mask = (conf_interp > 0.15) & (rms_norm > 0.12)
    amp = np.where(voiced_mask, 0.35 * rms_norm, 0.0)
    phase = (2.0 * np.pi * np.cumsum(freq_interp)) / sr
    return (amp * (0.8 * np.sin(phase) + 0.2 * np.sin(2.0 * phase))).astype(np.float32)

def process_word(word_id, mp3_path, visual_path):
    # Locate audio file (check local path, /content/, or /content/audio/)
    resolved_path = mp3_path
    if not os.path.exists(resolved_path):
        candidates = [f"audio/{word_id}.mp3", f"/content/{word_id}.mp3", f"/content/audio/{word_id}.mp3"]
        for c in candidates:
            if os.path.exists(c):
                resolved_path = c
                break

    y_raw, sr = librosa.load(resolved_path, sr=16000)
    y, _ = librosa.effects.trim(y_raw, top_db=28)
    y = np.pad(y, (int(0.04 * sr), int(0.04 * sr)), mode="constant")

    print(f"[{word_id}] Listening to audio ({len(y)/sr:.2f}s)...")

    # STEP 1: Acoustic Forced Alignment to Phonemes (MMS_FA)
    lexicon = WORD_PHONEME_LEXICON.get(word_id, [
        {"ipa": "a", "kind": "vowel", "char": "a"}
    ])
    fa_transcript = "".join(item["char"] for item in lexicon)

    waveform = torch.tensor(y).unsqueeze(0).to(device)
    with torch.inference_mode():
        emission, _ = fa_model(waveform)
    
    tokens = fa_tokenizer([fa_transcript])
    token_spans = None
    try:
        token_spans = fa_aligner(emission[0], tokens[0])
    except Exception as e:
        print(f"  -> Aligner note for {word_id}: {e}")

    ratio = waveform.size(1) / emission.size(1)
    phonemes_out = []

    # If alignment succeeded and matched expected count, use precise spans
    if token_spans and len(token_spans) == len(lexicon):
        for idx, (span, spec) in enumerate(zip(token_spans, lexicon)):
            s_idx = max(0, int(span.start * ratio) - int(0.012 * sr))
            e_idx = min(len(y), int(span.end * ratio) + int(0.018 * sr))
            clip = apply_fade(y[s_idx:e_idx], sr=sr, fade_ms=8)
            clip_path = f"audio/slices/{word_id}_p{idx+1}.wav"
            sf.write(clip_path, clip, sr)
            phonemes_out.append({
                "id": f"{word_id}_p{idx+1}",
                "kind": spec["kind"],  # "consonant" or "vowel"
                "audio_url": spec.get("ref_audio", clip_path)
            })
            print(f"  -> Phoneme {idx+1}: /{spec['ipa']}/ ({spec['kind']}) [{s_idx/sr:.2f}s - {e_idx/sr:.2f}s] -> {clip_path}")
    else:
        # Acoustic boundary division proportional to phonetic timing
        print(f"  -> Using acoustic frame distribution for {word_id}")
        dur_total = len(y)
        n = len(lexicon)
        for idx, spec in enumerate(lexicon):
            s_idx = int((idx / n) * dur_total)
            e_idx = int(((idx + 1) / n) * dur_total)
            clip = apply_fade(y[s_idx:e_idx], sr=sr, fade_ms=8)
            clip_path = f"audio/slices/{word_id}_p{idx+1}.wav"
            sf.write(clip_path, clip, sr)
            phonemes_out.append({
                "id": f"{word_id}_p{idx+1}",
                "kind": spec["kind"],
                "audio_url": spec.get("ref_audio", clip_path)
            })
            print(f"  -> Phoneme {idx+1}: /{spec['ipa']}/ ({spec['kind']}) -> {clip_path}")

    # STEP 2: Syllable Duration Segmentation
    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=160)[0]
    peaks, _ = find_peaks(rms, distance=12, prominence=np.max(rms) * 0.2)
    if len(peaks) == 0:
        peaks = [len(rms) // 2]
    boundaries = [0] + [int((peaks[i] + peaks[i+1]) / 2) for i in range(len(peaks) - 1)] + [len(rms)]
    syllables_out = []
    for idx in range(len(boundaries) - 1):
        s_s, e_s = (boundaries[idx] * 160) / sr, (boundaries[idx+1] * 160) / sr
        dur = max(0.08, e_s - s_s)
        clip_path = f"audio/slices/{word_id}_s{idx+1}.wav"
        sf.write(clip_path, apply_fade(y[int(s_s * sr):int(e_s * sr)], sr=sr, fade_ms=12), sr)
        syllables_out.append({
            "id": f"{word_id}_s{idx+1}",
            "width_px": int(min(180, max(75, dur * 240))),
            "audio_url": clip_path
        })

    # STEP 3: TorchCREPE F0 Pitch Contour & Humming Synthesis
    pitch, periodicity = torchcrepe.predict(
        torch.tensor(y).unsqueeze(0), sr, hop_length=160, fmin=65.0, fmax=500.0,
        model="full", decoder=torchcrepe.decode.viterbi, return_periodicity=True, device=device
    )
    frequency = pitch.squeeze(0).cpu().numpy()
    conf = periodicity.squeeze(0).cpu().numpy()
    hum_out = f"audio/slices/{word_id}_hum.wav"
    sf.write(hum_out, synthesize_humming_track(y, frequency, conf, sr=sr), sr)

    tones_out = []
    for idx in range(len(syllables_out)):
        s_f = int((idx / len(syllables_out)) * len(frequency))
        e_f = int(((idx + 1) / len(syllables_out)) * len(frequency))
        slope = float(frequency[e_f-1] - frequency[s_f]) if e_f - s_f > 1 else 0.0
        style = "solid" if slope > 12 else ("ring" if slope < -12 else "dot")
        tones_out.append({"id": f"{word_id}_t{idx+1}", "style": style, "audio_url": hum_out})

    return {
        "id": word_id,
        "visual_url": visual_path,
        "full_audio_url": f"audio/{word_id}.mp3",
        "humming_audio_url": hum_out,
        "phonemes": phonemes_out,
        "syllables": syllables_out,
        "tones": tones_out
    }

# 4. Run on Level 0 words
words = [
    ("cat", "audio/cat.mp3", "images/visuals/cat.jpg"),
    ("dog", "audio/dog.mp3", "images/visuals/dog.jpg"),
    ("walk", "audio/walk.mp3", "images/visuals/walk.gif"),
    ("jump", "audio/jump.mp3", "images/visuals/jump.gif"),
    ("tree", "audio/tree.mp3", "images/visuals/tree.jpg")
]

dataset_items = []
for w, p, v in words:
    resolved_p = p if os.path.exists(p) else (f"/content/{w}.mp3" if os.path.exists(f"/content/{w}.mp3") else f"/content/audio/{w}.mp3")
    if os.path.exists(resolved_p):
        dataset_items.append(process_word(w, resolved_p, v))
    else:
        print(f"Warning: audio for '{w}' not found at {p}")

dataset = {"items": dataset_items}

with open("lesson_data.json", "w") as f:
    json.dump(dataset, f, indent=2)

print(f"\nSuccessfully processed {len(dataset_items)} words into lesson_data.json!")

# 5. Zip lesson, assessment, and audio assets
with zipfile.ZipFile("gll_update_assets.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("lesson_data.json")
    if os.path.exists("assessment_data.json"):
        zf.write("assessment_data.json")
    for folder in ["audio/slices", "audio/phonemes", "audio/test_bank"]:
        if not os.path.isdir(folder):
            continue
        for root, _, files in os.walk(folder):
            for file in files:
                file_path = os.path.join(root, file)
                zf.write(file_path)

print("\nDone! Download gll_update_assets.zip and extract it into your project root.")
